# exp4_graded_precision — Axis 3, graded perception

La précision ρ de la perception des émotions varie sur {0 ; 0,25 ; 0,5 ; 0,75 ; 1},
croisée avec $w_e$ ∈ {0 ; 0,5} (méthode journal, *Axis 3*). Le croisement sépare la valeur
informative des émotions pour des agents égoïstes de leur rôle pour des agents
empathiques. Les conditions à ρ = 0 et ρ = 1 sont les conditions A à D de `exp1` : mêmes
runs, calculés une seule fois.

Analyses : $r_s$ de Spearman et Jonckheere–Terpstra sur ρ pour chaque $w_e$, puis ANOVA à
deux facteurs ρ × $w_e$ pour l'interaction. L'unité d'analyse est le run.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

EXPERIMENT = "exp4_graded_precision"
RESULTS_DIR = Path("..") / "csv_results" / EXPERIMENT
LAST_N_EVAL = 10
ALPHA_LEVEL = 0.05
METRICS = ["depletion_step", "gini_meals", "total_meals"]

## 1. Chargement et contrôles

In [ ]:
def load_runs(folder):
    """evaluation episodes of the finished runs of one experiment folder."""
    files = sorted(Path(folder).glob("*_episode_summary.csv"))
    if not files:
        raise FileNotFoundError(f"no *_episode_summary.csv under {Path(folder).resolve()}")
    frames = [pd.read_csv(p).assign(source_file=p.name) for p in files]
    df = pd.concat(frames, ignore_index=True)
    last = df[df.phase == "train"].groupby("source_file").episode.max()
    unfinished = last.index[last < last.max()]
    if len(unfinished):
        print(f"{len(unfinished)} unfinished run(s) dropped: {list(unfinished)}")
        df = df[~df.source_file.isin(unfinished)]
    df["depletion_step"] = df.total_steps
    df["censored"] = (df.total_steps >= df.max_steps) & (df.resource_remaining > 0)
    df["total_meals"] = df[[f"meals_{i}" for i in range(6)]].sum(axis=1)
    return df[df.phase == "eval"].copy()


def run_values(ev, keys):
    """one value per run: its mean over the last LAST_N_EVAL evaluation episodes."""
    tail = ev.sort_values("episode").groupby(keys + ["seed"], observed=True).tail(LAST_N_EVAL)
    return (tail.groupby(keys + ["seed"], observed=True)
                .agg(depletion_step=("depletion_step", "mean"), gini_meals=("gini_meals", "mean"),
                     total_meals=("total_meals", "mean"), censored=("censored", "mean"))
                .reset_index())


ev = load_runs(RESULTS_DIR)
dupes = int(ev.duplicated(["condition", "seed", "episode"]).sum())
print("duplicated evaluation rows:", dupes)
runs = run_values(ev, ["condition", "emotion_precision", "alpha"]).rename(columns={"emotion_precision": "rho"})
display(runs.groupby(["alpha", "rho"]).seed.nunique().unstack("rho"))
print("censoring rate (above 10 %: complement with a survival analysis, see the method)")
display(runs.groupby(["alpha", "rho"]).censored.mean().unstack("rho").round(3))
display(runs.groupby(["alpha", "rho"])[METRICS].agg(["mean", "std"]).round(3))

## 2. Tendance sur ρ, pour chaque $w_e$

In [ ]:
def spearman_ci(rho: float, n: int, conf: float = 0.95) -> tuple[float, float]:
    """Fisher-z CI for Spearman's rho, with the Bonett-Wright variance correction."""
    if n < 6 or not np.isfinite(rho) or abs(rho) >= 1:
        return (np.nan, np.nan)
    delta = stats.norm.ppf(1 - (1 - conf) / 2) * np.sqrt((1 + rho ** 2 / 2) / (n - 3))
    return float(np.tanh(np.arctanh(rho) - delta)), float(np.tanh(np.arctanh(rho) + delta))


def jonckheere_terpstra(groups: list[np.ndarray]) -> tuple[float, float, float]:
    """Ordered-alternatives trend test (normal approximation, ties at 0.5).

    `groups` must be given in the hypothesised increasing order. Unlike
    Kruskal-Wallis this uses the ordering of alpha, so it is the right rank test
    for a dose-response claim.
    """
    k = len(groups)
    jt = 0.0
    for i in range(k):
        for j in range(i + 1, k):
            a = np.asarray(groups[i])[:, None]
            b = np.asarray(groups[j])[None, :]
            jt += float((a < b).sum() + 0.5 * (a == b).sum())
    ns = np.array([len(g) for g in groups], dtype=float)
    total = ns.sum()
    mean = (total ** 2 - (ns ** 2).sum()) / 4
    var = (total ** 2 * (2 * total + 3) - (ns ** 2 * (2 * ns + 3)).sum()) / 72
    z = (jt - mean) / np.sqrt(var)
    return jt, float(z), float(2 * stats.norm.sf(abs(z)))


rows = []
for alpha, group in runs.groupby("alpha"):
    levels = sorted(group.rho.unique())
    for metric in METRICS:
        rho_s, p_s = stats.spearmanr(group.rho, group[metric])
        low, high = spearman_ci(rho_s, len(group))
        samples = [group.loc[group.rho == r, metric].to_numpy() for r in levels]
        rows.append({"alpha": alpha, "metric": metric, "n": len(group), "r_s": rho_s, "ci_low": low,
                     "ci_high": high, "p_spearman": p_s, "p_jonckheere": jonckheere_terpstra(samples)[-1]})
display(pd.DataFrame(rows).round(4))

## 3. ANOVA ρ × $w_e$

In [ ]:
for metric in METRICS:
    fit = ols(f"{metric} ~ C(rho) * C(alpha)", data=runs).fit()
    table = anova_lm(fit, typ=2)
    table["partial_eta_sq"] = table.sum_sq / (table.sum_sq + table.loc["Residual", "sum_sq"])
    table.loc["Residual", "partial_eta_sq"] = np.nan
    print(metric)
    display(table.round(4))
    cells = [g[metric].to_numpy() for _, g in runs.groupby(["rho", "alpha"])]
    print(f"  Shapiro (residuals) p = {stats.shapiro(fit.resid).pvalue:.3f}, "
          f"Levene p = {stats.levene(*cells).pvalue:.3f}, "
          f"Kruskal-Wallis p = {stats.kruskal(*cells).pvalue:.4f}\n")